# 01 · Graphs Live in Your Database

**Apache AGE** turns the PostgreSQL you already run into a graph database. No second
system, no copy, no ETL — Cypher runs in the *same* database as your tables.

This first notebook sets up the mental model: connect, see a Cypher query next to the
same data queried in Python, and fix the 101 vocabulary — **nodes, edges, properties,
labels**. The one modelling decision behind every graph: *what is a node, what is an
edge, what is a property.*

Figures are saved to `outputs/figures/*.png`; numbers to `outputs/data/*.json`.

In [1]:
import sys
from pathlib import Path

# Make `src` importable no matter where the notebook is executed from.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json

from src.age.connect import age_available, get_connection
from src.age.cypher import run_cypher
from src.config import load_settings
from src.datasets.graphs import KNOWS, all_graphs
from src.outputs import figure_path, save_json
from src.queries.traversal import reachable_networkx
from src.visualisation import draw_graph

settings = load_settings()
AGE = age_available(settings)
print("AGE usable in this database:", AGE)

# The graphs were loaded by `make load`; we also build them in Python for the baselines.
graphs = all_graphs(settings)
social = graphs["social_txn"]
print(f"social_txn: {len(social.nodes)} nodes, {len(social.edges)} edges, "
      f"{len(social.rings)} planted rings")

AGE usable in this database: True
social_txn: 400 nodes, 3576 edges, 6 planted rings


## Cypher next to the same data, in one database

The extension is enabled once with `CREATE EXTENSION age` (done by `scripts/init_db.sql`).
Then every connection does two lines — `LOAD 'age'` and set the `search_path` — which is
exactly what `src/age/connect.py` centralises. Here is a first Cypher query, and the same
count computed from the Python graph: the graph and the relational data are the *same
records*.

In [2]:
conn = get_connection(settings) if AGE else None

if AGE:
    rows = run_cypher(conn, settings.graphs.social,
                      "MATCH (a:Account) RETURN count(*)", columns=("n",))
    age_count = int(rows[0][0])
else:
    age_count = None  # baselines still teach the lesson without a server

py_count = len(social.nodes)
print("Accounts — via Cypher/AGE:", age_count)
print("Accounts — via Python graph:", py_count)
assert age_count in (None, py_count)

Accounts — via Cypher/AGE: 400
Accounts — via Python graph: 400


## The 101 vocabulary

A **node** (vertex) has a **label** (`Account`) and **properties** (`name`, `risk`). An
**edge** (relationship) also has a label (`KNOWS`, `PAYS`) and can carry properties
(`amount`). Let's tally the labels and save the summary.

In [3]:
node_labels = {}
for n in social.nodes:
    node_labels[n.label] = node_labels.get(n.label, 0) + 1

edge_labels = {}
for e in social.edges:
    edge_labels[e.label] = edge_labels.get(e.label, 0) + 1

summary = {
    "graph": "social_txn",
    "nodes": len(social.nodes),
    "edges": len(social.edges),
    "node_labels": node_labels,
    "edge_labels": edge_labels,
    "planted_rings": len(social.rings),
    "age_account_count": age_count,
    "sample_node_properties": social.nodes[0].props,
    "sample_edge_properties": next(e.props for e in social.edges if e.label == "PAYS"),
}
print(json.dumps(summary, indent=2))
print("saved ->", save_json("01_summary.json", summary))

{
  "graph": "social_txn",
  "nodes": 400,
  "edges": 3576,
  "node_labels": {
    "Account": 400
  },
  "edge_labels": {
    "KNOWS": 3018,
    "PAYS": 558
  },
  "planted_rings": 6,
  "age_account_count": 400,
  "sample_node_properties": {
    "name": "acct_0",
    "risk": 0.324
  },
  "sample_edge_properties": {
    "amount": 1698
  }
}
saved -> /workspaces/apache_age_101/outputs/data/01_summary.json


## See a neighbourhood

A picture of the `KNOWS` neighbourhood around one account (up to 2 hops). This is the
shape AGE will let us query in one line in the next notebook.

In [4]:
ego_ids = reachable_networkx(social, 0, 1, 2, KNOWS) | {0}
ego = social.to_networkx(KNOWS).subgraph(ego_ids)
path = draw_graph(
    ego,
    figure_path("01_knows_ego.png"),
    title="KNOWS neighbourhood of account 0 (<= 2 hops)",
    highlight={0},
)
print(f"{ego.number_of_nodes()} nodes, {ego.number_of_edges()} edges -> {path}")

55 nodes, 220 edges -> /workspaces/apache_age_101/outputs/figures/01_knows_ego.png


In [5]:
if conn is not None:
    conn.close()
print("Done. Next: 02 — the n-hop join problem.")

Done. Next: 02 — the n-hop join problem.


**Takeaway.** Graph and relational querying share one database. You enable the
extension once, then Cypher runs right next to your SQL. Next we show *why* you'd want to:
the multi-hop questions SQL answers badly.